# 02 — Eğitim ve değerlendirme (GPU)
**Çalışma zamanı → GPU** (T4, L4 veya A100; hangisi olduğu kurulum hücresinde yazar).
Secrets: `KAGGLE_API_TOKEN`, `GITHUB_TOKEN`.
**Kullanım:** ilk kod hücresinde `CONFIG` ve adım anahtarlarını ayarla, sonra
**Çalışma zamanı → Tümünü çalıştır**. Kapalı adımlar kendiliğinden atlanır.
Yeni deneyde sıra: smoke + 1 epoch (süre ölç) → Claude'a getir → `TRAIN` + `EVAL`.
Veri: private Kaggle veri seti `ouzhantarhan/baykus-dv-rgbt` (ADR 0004), birkaç dakikada iner.
Çıktılar Drive'da `MyDrive/baykus/{runs,eval,results}`. Her epoch `weights/last.pt` olarak
Drive'a yazılır. Oturum koparsa ayarlara dokunmadan yeniden **Tümünü çalıştır**: eğitim
hücresi yarım kalan eğitimi bulup kaldığı epoch'tan sürdürür.

In [ ]:
CONFIG = 'configs/e1_rgb.yaml'   # e2_ir, e3_early_fusion, e3c_modality_dropout
BRANCH = 'main'                  # haftalık dalı denemek için ör. 'h3'
# Adım anahtarları (True = çalışır, False = atlanır):
SMOKE = True    # 1) %5 veri, 1 epoch: hattın çalıştığını doğrular (yeni deneyde ilk iş)
EPOCH1 = True   # 2) 1 tam epoch: süre ölçümü (GPU bütçesi)
TRAIN = False   # 3) tam eğitim; yarım kalan varsa devam eder, biten varsa atlar
EVAL = False    # 4) değerlendirme (val): biten tam eğitimin best.pt'si
COMPARE = False # 5) E3 − E1, E3 − E2, E2 − E1 farkları (üç modelin de 4. adımı bitmiş olmalı)
DRIVE = '/content/drive/MyDrive/baykus'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
!pip -q install uv kaggle
!python --version   # Spike C: Colab görüntüsü zamanla değişir
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc              # CPU çekirdeği: veri yükleme hızını belirler (gpu-budget.md)
!uv pip install --system -q -e '/content/baykus/python[train]'
import sys  # editable kurulumun .pth'si çalışan çekirdekte okunmaz; checkpoint hücreleri için
sys.path.insert(0, '/content/baykus/python/src')
import yaml
_cfg = yaml.safe_load(open(f'/content/baykus/{CONFIG}', encoding='utf-8'))
MODE, EXP = _cfg['mode'], _cfg['experiment']
print('MODE =', MODE, '· EXP =', EXP)

In [ ]:
%%time
# İşlenmiş veri: private Kaggle veri seti (ADR 0004). Kaggle yüklenen tar'ları klasöre açtı;
# CLI klasör seçerek indiremediği için veri setinin tamamı iner (~9,8 GB).
DS = 'ouzhantarhan/baykus-dv-rgbt'
!kaggle datasets download {DS} -p /content/dv_rgbt --unzip -q
!mv /content/dv_rgbt/meta /content/meta
!baykus-data illum-relabel --meta /content/meta   # ADR 0005: Kaggle'daki etiketler eski eşikle
!baykus-data slices --out /content/dv_rgbt --meta /content/meta
!baykus-data yaml --out /content/dv_rgbt
!du -sh /content/dv_rgbt/* /content/meta

In [ ]:
from pathlib import Path
NEED = {'rgb': ['rgb'], 'ir': ['ir'], 'early_fusion': ['rgb', 'ir']}[MODE]
# H1 Görev 10'dan taşındı: kutular araçların üstüne oturuyor mu? (gözle)
# Her set kendi görüntüsünde: solda RGB etiketleri RGB'de, sağda IR etiketleri IR'de.
import random, cv2, numpy as np
from IPython.display import Image, display
random.seed(1)
root = Path('/content/dv_rgbt')

def boxed(img_path, label_path):
    im = cv2.imread(str(img_path), cv2.IMREAD_COLOR)  # IR gri → 3 kanal
    h_im, w_im = im.shape[:2]
    lines = label_path.read_text(encoding='utf-8').splitlines() if label_path.exists() else []
    for ln in lines:
        c, cx, cy, w, h = map(float, ln.split())
        cv2.rectangle(im, (int((cx - w / 2) * w_im), int((cy - h / 2) * h_im)),
                      (int((cx + w / 2) * w_im), int((cy + h / 2) * h_im)), (0, 255, 0), 1)
    return im, len(lines)

for s in random.sample(sorted(p.stem for p in (root / 'rgb/images/train').glob('*.jpg')), 3):
    rgb, n_rgb = boxed(root / f'rgb/images/train/{s}.jpg', root / f'labels_rgb/train/{s}.txt')
    ir, n_ir = boxed(root / f'ir/images/train/{s}.png', root / f'labels_ir/train/{s}.txt')
    print(f'{s}: RGB {n_rgb} kutu, IR {n_ir} kutu')
    cv2.imwrite(f'/content/check_{s}.jpg', np.hstack([rgb, ir]))
    display(Image(f'/content/check_{s}.jpg'))
for d in NEED:  # beklenen düzen: <modalite>/images/{train,val,test} (Data Explorer'da görüldü)
    for split in ('train', 'val', 'test'):
        assert Path(f'/content/dv_rgbt/{d}/images/{split}').is_dir(), f"{d}/images/{split} yok; ls çıktısını Claude'a getir"

In [ ]:
%%time
# ADR 0004: rgbt4 Kaggle'da yok; her E3 oturumunda rgb + ir'den üretilir (sadece train + val).
# Süreyi Claude'a getir (ADR 0004'e yazılır).
import os
if MODE == 'early_fusion':
    NPROC = os.cpu_count()
    !baykus-data rgbt4 --out /content/dv_rgbt --splits train val --workers {NPROC}
    !du -sh /content/dv_rgbt/rgbt4
    !df -h /content

In [ ]:
%%time
# 1) smoke: %5 veri, 1 epoch — hattın uçtan uca çalıştığını doğrular.
# Bu deneyin biten smoke'u varsa tekrar koşmaz (yeniden ölçmek için Drive'daki klasörü sil).
from baykus.train.checkpoints import find_trial_runs
_prev = find_trial_runs(f'{DRIVE}/runs', EXP, 'smoke')
if SMOKE and _prev:
    print('smoke zaten var, atlandı:', _prev[-1].name)
elif SMOKE:
    !cd /content/baykus/python && baykus-train --config ../{CONFIG} --data-root /content/dv_rgbt --project {DRIVE}/runs --results {DRIVE}/results --smoke
else:
    print('SMOKE = False → atlandı')

In [ ]:
%%time
# 2) 1 tam epoch — süresi GPU bütçesine yazılır (spec §10.2). Biten varsa tekrar koşmaz.
from baykus.train.checkpoints import find_trial_runs
_prev = find_trial_runs(f'{DRIVE}/runs', EXP, 'ep1')
if EPOCH1 and _prev:
    print('1 epoch ölçümü zaten var, atlandı:', _prev[-1].name)
elif EPOCH1:
    !cd /content/baykus/python && baykus-train --config ../{CONFIG} --data-root /content/dv_rgbt --project {DRIVE}/runs --results {DRIVE}/results --epochs 1
else:
    print('EPOCH1 = False → atlandı')

🚧 **KAPI:** Yeni bir deneyde 1) ve 2) bitince süreleri Claude'a getir; bütçe onaylanınca ilk hücrede `SMOKE = False`, `EPOCH1 = False`, `TRAIN = True`, `EVAL = True` yapıp yeniden Tümünü çalıştır.

In [ ]:
%%time
# 3) tam eğitim (config'teki epoch sayısı). Checkpoint: her epoch sonunda
# {DRIVE}/runs/<run>/weights/last.pt ve 5 epoch'ta bir epochN.pt Drive'a yazılır.
# Yarım kalan eğitim varsa kaldığı epoch'tan sürer; biten varsa yeniden eğitmez.
from baykus.train.checkpoints import find_runs, status_report
if TRAIN:
    todo = find_runs(f'{DRIVE}/runs', EXP, finished=False)
    done = find_runs(f'{DRIVE}/runs', EXP, finished=True)
    if todo:
        RUN_DIR = str(todo[-1])
        print('yarım kalan eğitim, devam:', RUN_DIR)
        !cd /content/baykus/python && baykus-train --config ../{CONFIG} --resume "{RUN_DIR}" --data-root /content/dv_rgbt --results {DRIVE}/results
    elif done:
        print('bu deneyin biten tam eğitimi var, yeniden eğitilmedi:', done[-1])
    else:
        !cd /content/baykus/python && baykus-train --config ../{CONFIG} --data-root /content/dv_rgbt --project {DRIVE}/runs --results {DRIVE}/results
else:
    print('TRAIN = False → atlandı')
print(status_report(f'{DRIVE}/runs', EXP))  # 📌 Drive'da hangi epoch'a kadar kayıtlı

In [ ]:
# 4) değerlendirme (val): bu config'in biten en son tam eğitiminin best.pt'si
from baykus.train.checkpoints import find_runs
if EVAL:
    done = [str(p) for p in find_runs(f'{DRIVE}/runs', EXP, finished=True)]
    assert done, 'biten tam eğitim yok; önce 3) bitmeli'
    W = f'{done[-1]}/weights/best.pt'
    print(W)
    !cd /content/baykus/python && baykus-eval run --config ../{CONFIG} --weights "{W}" --data-root /content/dv_rgbt --meta /content/meta --project {DRIVE}/eval --results {DRIVE}/results
else:
    print('EVAL = False → atlandı')

In [ ]:
# 5) eşleştirilmiş karşılaştırma (val, IR etiketleri; FR-V8): A − B, aydınlatma kırılımıyla.
# Biten eğitimi ya da değerlendirmesi olmayan deney atlanır; var olan karşılaştırma tekrar yapılmaz.
# H4'ün karşılaştırmaları (E3c, E4) 04_fusion'da.
import json
from pathlib import Path
from baykus.train.checkpoints import find_runs

def preds(exp):
    runs = find_runs(f'{DRIVE}/runs', exp, finished=True)
    if not runs:
        return None
    run_id = json.loads((runs[-1] / 'baykus_train.json').read_text(encoding='utf-8'))['run_id']
    p = Path(f'{DRIVE}/eval/{run_id}_val/predictions_coco.json.gz')
    return p if p.exists() else None

if COMPARE:
    for A, B in [('e3_early_fusion', 'e1_rgb'), ('e3_early_fusion', 'e2_ir'), ('e2_ir', 'e1_rgb')]:
        name = f'{A}-vs-{B}_val_ir'
        if Path(f'{DRIVE}/results/compare/{name}.json').exists():
            print('zaten var:', name)
            continue
        PA, PB = preds(A), preds(B)
        if PA is None or PB is None:
            print('atlandı (biten eğitim ya da değerlendirme yok):', name)
            continue
        !cd /content/baykus/python && baykus-eval compare --gt /content/dv_rgbt/coco/val_ir.json --a "{PA}" --b "{PB}" --slices /content/meta/slices_val.json --split val --name {name} --results {DRIVE}/results
else:
    print('COMPARE = False → atlandı')